# ResNet50 — Training & Testing Accuracy Evaluation

Loads `checkpoint_resnet50.pth` and evaluates on:
1. **Training set** — measures how well the model fits training data
2. **Test set** — measures generalisation to unseen speakers

Plots: training history (from checkpoint), accuracy bar chart, distance distributions.

---
## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

print("All libraries loaded.")

## 2. Audio Transforms

Deterministic **center crop** for reproducible evaluation.

In [ ]:
TARGET_SR     = 16000
TARGET_LENGTH = TARGET_SR * 3   # 48000 samples

def crop_or_pad(audio):
    length = len(audio)
    if length > TARGET_LENGTH:
        start = (length - TARGET_LENGTH) // 2   # center crop
        audio = audio[start:start + TARGET_LENGTH]
    elif length < TARGET_LENGTH:
        audio = np.pad(audio, (0, TARGET_LENGTH - length), mode='constant')
    return audio

mel_transform   = T.MelSpectrogram(sample_rate=16000, n_fft=400, hop_length=160, n_mels=80)
amplitude_to_db = T.AmplitudeToDB()
print("Transforms ready.")

## 3. SpeakerPairDataset

In [ ]:
class SpeakerPairDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.df              = dataframe
        self.mel_transform   = mel_transform
        self.amplitude_to_db = amplitude_to_db

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio)
        audio = torch.tensor(audio).float().unsqueeze(0)
        return self.amplitude_to_db(self.mel_transform(audio))

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        mel1  = self.load_audio(row["path1_abs"])
        mel2  = self.load_audio(row["path2_abs"])
        label = torch.tensor(row["label"]).float()
        return mel1, mel2, label

## 4. Model Architecture — ResNetSpeaker (ResNet50)

ResNet50 uses **bottleneck blocks** outputting **2048 features** (vs 512 for ResNet18/34).
The embedding layer automatically handles this via `in_features`.

In [ ]:
class ResNetSpeaker(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features   # 2048 for ResNet50
        self.backbone.fc = nn.Identity()
        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features  = self.backbone(x)
        embedding = self.embedding(features)
        return F.normalize(embedding, p=2, dim=1)

## 5. Load Saved Checkpoint

Update `CHECKPOINT_PATH` to your exact Kaggle dataset path if needed.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = ResNetSpeaker(embedding_dim=256).to(device)

CHECKPOINT_PATH = "/kaggle/input/datasets/tahmidulislamomi/resnet-50-trained-configuration/checkpoint_resnet50.pth"

checkpoint    = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

loss_history = checkpoint.get("loss_history", [])
acc_history  = checkpoint.get("acc_history",  [])

print(f"Model loaded from epoch {checkpoint.get('epoch', '?') + 1}")
print(f"Training history available: {len(loss_history)} epochs")

## 6. Plot Training History

Recovered from the checkpoint — no re-training needed.

In [ ]:
if loss_history:
    epochs_range = range(1, len(loss_history) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("ResNet50 — Training History (from checkpoint)", fontsize=13, fontweight='bold')

    axes[0].plot(epochs_range, loss_history, marker='o', color='tomato', linewidth=2)
    axes[0].set_title("Contrastive Loss per Epoch")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_range, acc_history, marker='o', color='steelblue', linewidth=2)
    axes[1].set_title("Training Accuracy per Epoch")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_history_resnet50.png", dpi=150)
    plt.show()
    print("Saved → training_history_resnet50.png")
else:
    print("No training history found in checkpoint.")

## 7. Evaluate Function

Uses **cosine similarity > 0.5** to predict same/different speaker.

In [ ]:
def evaluate(model, loader, device, label_name="Set"):
    model.eval()
    correct = 0; total = 0
    same_dists, diff_dists = [], []

    with torch.no_grad():
        for mel1, mel2, labels in tqdm(loader, desc=f"Evaluating {label_name}"):
            mel1    = mel1.to(device)
            mel2    = mel2.to(device)
            labels  = labels.to(device)

            emb1 = model(mel1)
            emb2 = model(mel2)

            cosine_sim = F.cosine_similarity(emb1, emb2)
            preds      = (cosine_sim > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            d = F.pairwise_distance(emb1, emb2).cpu().tolist()
            for dist, lbl in zip(d, labels.cpu().tolist()):
                (same_dists if lbl == 1 else diff_dists).append(dist)

    acc = correct / total
    print(f"{label_name} Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    return acc, same_dists, diff_dists

## 8. Training Set Evaluation

In [ ]:
TRAIN_ROOT  = "/kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset"
TRAIN_AUDIO = os.path.join(TRAIN_ROOT, "train-clean-100", "train-clean-100")
TRAIN_CSV   = os.path.join(TRAIN_ROOT, "train_pairs.csv")

train_df = pd.read_csv(TRAIN_CSV)

def to_train_abs(rel):
    return os.path.join(TRAIN_AUDIO, rel.replace("train-clean-100/", "", 1))

train_df["path1_abs"] = train_df["audio_path_1"].apply(to_train_abs)
train_df["path2_abs"] = train_df["audio_path_2"].apply(to_train_abs)

train_loader = DataLoader(
    SpeakerPairDataset(train_df, mel_transform, amplitude_to_db),
    batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)

train_acc, train_same, train_diff = evaluate(model, train_loader, device, "Training Set")

## 9. Test Set Evaluation

In [ ]:
TEST_ROOT   = "/kaggle/input/datasets/tahmidulislamomi/ml-project-testing-dataset"
REAL_AUDIO  = os.path.join(TEST_ROOT, "test-clean")
TEST_CSV    = os.path.join(TEST_ROOT, "test_pairs.csv")

test_df = pd.read_csv(TEST_CSV)

def to_test_abs(rel):
    return os.path.join(REAL_AUDIO, rel)

test_df["path1_abs"] = test_df["audio_path_1"].apply(to_test_abs)
test_df["path2_abs"] = test_df["audio_path_2"].apply(to_test_abs)

test_loader = DataLoader(
    SpeakerPairDataset(test_df, mel_transform, amplitude_to_db),
    batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)

test_acc, test_same, test_diff = evaluate(model, test_loader, device, "Test Set")

## 10. Accuracy Comparison Bar Chart

In [ ]:
labels_bar = ["Training Accuracy", "Test Accuracy"]
values_bar = [train_acc, test_acc]
colors_bar = ["steelblue", "darkorange"]

plt.figure(figsize=(7, 5))
bars = plt.bar(labels_bar, values_bar, color=colors_bar, width=0.4, edgecolor='white')
for bar, val in zip(bars, values_bar):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val*100:.2f}%", ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.ylim(0, 1.1)
plt.ylabel("Accuracy")
plt.title("ResNet50 — Training vs Test Accuracy", fontsize=13, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("accuracy_comparison_resnet50.png", dpi=150)
plt.show()
print("Saved → accuracy_comparison_resnet50.png")

## 11. Distance Distribution — Training Set vs Test Set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("ResNet50 — Embedding Distance Distribution", fontsize=13, fontweight='bold')

for ax, same, diff, title in [
    (axes[0], train_same[:5000], train_diff[:5000], "Training Set"),
    (axes[1], test_same[:5000],  test_diff[:5000],  "Test Set"),
]:
    ax.hist(same, bins=60, alpha=0.6, color='steelblue', label='Same Speaker (label=1)')
    ax.hist(diff, bins=60, alpha=0.6, color='tomato',    label='Different Speaker (label=0)')
    ax.set_title(title)
    ax.set_xlabel("Euclidean Distance")
    ax.set_ylabel("Count")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("distance_distribution_resnet50.png", dpi=150)
plt.show()
print("Saved → distance_distribution_resnet50.png")

## 12. Summary

In [ ]:
print("=" * 40)
print(f"  Training Accuracy : {train_acc*100:.2f}%")
print(f"  Test Accuracy     : {test_acc*100:.2f}%")
print(f"  Generalisation Gap: {(train_acc - test_acc)*100:.2f}%")
print("=" * 40)
print("Saved graphs:")
print("  - training_history_resnet50.png")
print("  - accuracy_comparison_resnet50.png")
print("  - distance_distribution_resnet50.png")